# HyperTools 1.0 Release QC — Verification Notebook

This notebook is the manual QC pass for **HyperTools 1.0** (`dev-1.0-refactor`
branch). Run every cell top-to-bottom in a **fresh Google Colab runtime**.
Each feature area below has:

1. A markdown header with a `☐ Works as expected — notes:` line for your
   overall verdict on that section.
2. One or more code cells that exercise the feature with real, small data
   (no mocks) and produce a visible figure or printed output.
3. A `**Verify:**` line after each code cell describing exactly what to look
   for, with `☐ pass ☐ fail — notes:` for you to fill in inline.

Keep this notebook as the QC record — save a copy (File > Save a copy in
Drive) once you've filled in the checkboxes/notes.

**Scope:** every public `hyp.*` function, plus named 1.0 features:
autoencoders (#162), cross-module kwargs (#138), manip chaining (#274/#153),
`Pipeline` (#227/#161), `animate=` dict form (#154), plot kwarg passthrough
(#103/#206), colorbar/surface/density, data loaders (#273/#116), gensim text
(#198), and LSL streaming (#130).

## 0. Install

☐ Works as expected — notes:

In [ ]:
# Installs hypertools 1.0 (dev-1.0-refactor) with the interactive (plotly),
# torch (autoencoders), and gensim (Word2Vec text) extras.
# Two optional extras are NOT installed here and are noted where relevant below:
#   - lsl    (hyp.io.lsl_stream; needs a live LSL outlet, not runnable in plain Colab)
#   - kaggle (hyp.load('kaggle/...'); needs kagglehub + Kaggle API credentials)
!pip install -q "hypertools[interactive,torch,gensim] @ git+https://github.com/ContextLab/hypertools.git@dev-1.0-refactor"

import hypertools as hyp
print("hypertools version:", hyp.__version__)

**Verify:** Install completes with no errors and a version string prints (e.g. `1.0.0.dev0`). — ☐ pass ☐ fail — notes:

## 1. Core plotting (hyp.plot)

☐ Works as expected — notes:

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
arr = rng.normal(size=(60, 4)).cumsum(axis=0)
arr_list = [rng.normal(size=(50, 4)).cumsum(axis=0) for _ in range(3)]

# Single array, 3D (default ndims=3), matplotlib backend
hyp.plot(arr, backend='matplotlib', title='single array, 3D, matplotlib')

# List of arrays, 2D, matplotlib backend
hyp.plot(arr_list, ndims=2, backend='matplotlib', title='list of 3 arrays, 2D, matplotlib')

# List of arrays, 3D, plotly backend (interactive)
hyp.plot(arr_list, ndims=3, backend='plotly', title='list of 3 arrays, 3D, plotly')

**Verify:** Three figures render in order: a single 3D matplotlib trajectory, a 2D matplotlib plot with 3 colored trajectories, and an interactive 3D plotly plot with 3 colored trajectories. — ☐ pass ☐ fail — notes:

## 2. Reduce (hyp.reduce)

☐ Works as expected — notes:

In [ ]:
from hypertools.reduce.reduce import reduce as hyp_reduce

X = rng.normal(size=(80, 10))

for name in ['PCA', 'IncrementalPCA', 'UMAP', 'TSNE', 'FastICA', 'MDS']:
    out = hyp_reduce(X, reduce=name, ndims=2)
    print(f"{name:15s} -> shape {np.asarray(out).shape}")

# Mixture model as a reducer: returns (n_samples, ndims) membership proportions (GH #174)
proportions = hyp_reduce(X, reduce='GaussianMixture', ndims=3)
print(f"{'GaussianMixture':15s} -> shape {np.asarray(proportions).shape}, "
      f"row sums ~1: {np.asarray(proportions).sum(axis=1)[:3].round(3)}")

# return_model=True + reuse on new (held-out) data
reduced, model = hyp_reduce(X, reduce='PCA', ndims=2, return_model=True)
X_new = rng.normal(size=(15, 10))
reduced_new = model.transform(X_new)
print("PCA return_model reuse -> new-data shape:", np.asarray(reduced_new).shape)

**Verify:** Every reducer name (incl. GaussianMixture-as-reducer with proportions summing to ~1) runs without error, and the reused fitted PCA model transforms new (15, 10) data to (15, 2). — ☐ pass ☐ fail — notes:

## 3. Autoencoders (hyp.reduce, GH #162)

☐ Works as expected — notes:

In [ ]:
# Shallow Autoencoder and VariationalAutoencoder: small epochs for speed, real torch training
ae_out = hyp_reduce(X, reduce={'model': 'Autoencoder',
                                'kwargs': {'n_components': 2, 'epochs': 15, 'verbose': True}})
print("Autoencoder output shape:", np.asarray(ae_out).shape)

vae_out = hyp_reduce(X, reduce={'model': 'VariationalAutoencoder',
                                 'kwargs': {'n_components': 2, 'epochs': 15}})
print("VariationalAutoencoder output shape:", np.asarray(vae_out).shape)

**Verify:** Autoencoder prints per-epoch progress (verbose=True); both output shapes are (80, 2) with no torch import errors. — ☐ pass ☐ fail — notes:

## 4. Align (hyp.align, GH #227)

☐ Works as expected — notes:

In [ ]:
from hypertools.align.align import align as hyp_align

# 3 noisy copies of the same underlying signal
base = rng.normal(size=(40, 5))
datasets = [base + rng.normal(scale=0.1, size=(40, 5)) for _ in range(3)]

aligned_hyper = hyp_align(datasets, model='HyperAlign')
print("HyperAlign shapes:", [np.asarray(a).shape for a in aligned_hyper])

aligned_srm, srm_model = hyp_align(datasets, model='SharedResponseModel', return_model=True)
print("SRM shapes:       ", [np.asarray(a).shape for a in aligned_srm])

# Reuse the fitted SRM Aligner on NEW data. Per GH #227 an alignment can only be
# reapplied to a list with the SAME number of datasets / columns as it was fit on
# (here: 3 datasets of 40x5). Passing a different structure raises a clear ValueError.
new_datasets = [base + rng.normal(scale=0.1, size=(40, 5)) for _ in range(3)]
aligned_new = srm_model.transform(new_datasets)
print("Reused SRM transform on 3 new datasets -> shapes:", [np.asarray(a).shape for a in aligned_new])

**Verify:** HyperAlign and SRM both return 3 arrays shaped (40, 5); the reused fitted SRM model applies to new data via .transform without refitting. — ☐ pass ☐ fail — notes:

## 5. Cluster (hyp.cluster)

☐ Works as expected — notes:

In [ ]:
from hypertools.cluster.cluster import cluster as hyp_cluster

Xc = np.vstack([
    rng.normal(loc=[0, 0], scale=0.3, size=(30, 2)),
    rng.normal(loc=[5, 5], scale=0.3, size=(30, 2)),
    rng.normal(loc=[0, 5], scale=0.3, size=(30, 2)),
])

labels = hyp_cluster(Xc, cluster='KMeans', n_clusters=3)
print("KMeans labels (first 10):", np.asarray(labels)[:10])
hyp.plot(Xc, ndims=2, hue=labels, title='KMeans hard clusters')

# Mixture model = soft clustering; pass proportions straight to hue for blended colors
soft_props = hyp_cluster(Xc, cluster='GaussianMixture', n_clusters=3)
hyp.plot(Xc, ndims=2, hue=np.asarray(soft_props), title='GaussianMixture soft-cluster coloring')

**Verify:** The first plot shows 3 discretely-colored blobs (hard KMeans labels); the second shows the same blobs with blended colors near boundaries (soft GaussianMixture membership). — ☐ pass ☐ fail — notes:

## 6. Manip + chaining (hyp.manip, GH #274/#153)

☐ Works as expected — notes:

In [ ]:
from hypertools.manip.manip import manip as hyp_manip

Xt = np.cumsum(rng.normal(size=(200, 3)), axis=0)

# Chained manip list (the canonical docstring example): Smooth -> Resample -> ZScore
chained = hyp_manip(Xt, model=[
    {'model': 'Smooth', 'kwargs': {'kernel_width': 25}},
    {'model': 'Resample', 'kwargs': {'n_samples': 1000}},
    'ZScore',
])
print("Chained manip output shape (expect (1000, 3), resampled+z-scored):", np.asarray(chained).shape)
hyp.plot(chained, ndims=2, title='Smooth -> Resample -> ZScore chain')

# Compare the three Smooth kernels on the same noisy signal.
# (Use the string+kwargs form: hyp.manip(x, model='Smooth', kernel=..., kernel_width=...).)
import matplotlib.pyplot as plt

noisy = np.cumsum(rng.normal(size=(150, 1)), axis=0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(noisy, color='lightgray', label='raw', linewidth=1)
for kernel in ['savgol', 'gaussian', 'boxcar']:
    smoothed = hyp_manip(noisy, model='Smooth', kernel=kernel, kernel_width=15)
    ax.plot(np.asarray(smoothed), label=kernel)
ax.legend()
ax.set_title('Smooth kernel comparison')
plt.show()

**Verify:** Chained output is (1000, 3) and visibly smoothed/resampled; the kernel-comparison plot shows all 3 smoothed curves (savgol/gaussian/boxcar) tracking but smoother than the gray raw line. — ☐ pass ☐ fail — notes:

## 7a. Normalize, analyze, apply_model (one real call each)

☐ Works as expected — notes:

In [ ]:
from hypertools.tools.normalize import normalize as hyp_normalize
from hypertools.tools.analyze import analyze as hyp_analyze
from hypertools.core.model import apply_model

# normalize (return_model). NOTE: reusing the fitted normalizer on new data via
# .transform() currently raises IndexError (flagged in the QC task list, P0-1);
# here we just confirm the forward normalization centers the data.
Xn = rng.normal(loc=5, scale=3, size=(50, 4))
normed, norm_model = hyp_normalize(Xn, normalize='across', return_model=True)
print("normalize: mean~0?", np.asarray(normed).mean().round(3), "| fitted model:", type(norm_model).__name__)

# analyze: normalize -> reduce -> cluster in one call, return_model=True
Xa = rng.normal(size=(60, 8))
analyzed, analyze_model = hyp_analyze(
    Xa, normalize='across', reduce='PCA', ndims=2, cluster='KMeans', return_model=True)
print("analyze output shape:", np.asarray(analyzed).shape,
      "| pipeline steps:", [name for name, _ in analyze_model.steps])

# apply_model: apply a named model directly, return_model=True
Xam = rng.normal(size=(40, 6))
transformed, fitted_model = apply_model(Xam, model='PCA', return_model=True, ndims=3)
print("apply_model('PCA') shape:", np.asarray(transformed).shape, "| model type:", type(fitted_model).__name__)

**Verify:** normalize/analyze/apply_model all run; analyze's fitted Pipeline prints named steps (normalize->reduce->cluster order); apply_model returns a fitted PCA instance. — ☐ pass ☐ fail — notes:

## 7b. Describe, predict, impute (one real call each)

☐ Works as expected — notes:

In [ ]:
from hypertools.reduce.describe import describe as hyp_describe
from hypertools.predict.predict import predict as hyp_predict
from hypertools.impute.impute import impute as hyp_impute

# describe: correlation-vs-dimensions plot
Xd = rng.normal(size=(50, 12))
desc_result = hyp_describe(Xd, reduce='PCA', max_dims=10, show=True)
print("describe() keys:", list(desc_result.keys()))

# predict: forecast t new rows
Xp = np.cumsum(rng.normal(size=(40, 2)), axis=0)
forecast = hyp_predict(Xp, model='Kalman', t=10)
print("forecast shape (expect 10 rows):", np.asarray(forecast).shape)
hyp.plot([Xp, forecast], ndims=2, legend=['observed', 'forecast'], title='Kalman forecast')

# impute: fill NaNs
Xi = rng.normal(size=(30, 5))
Xi_missing = Xi.copy()
Xi_missing[rng.random((30, 5)) < 0.1] = np.nan
print("NaNs before/after impute:", np.isnan(Xi_missing).sum(), "->",
      np.isnan(np.asarray(hyp_impute(Xi_missing, model='PPCA'))).sum())

**Verify:** describe() shows a correlation-vs-dimensions plot with 'average'/'individual' keys; predict()'s forecast plausibly continues the trend; impute() reduces the NaN count to 0. — ☐ pass ☐ fail — notes:

## 8. Pipeline (hyp.Pipeline, GH #227/#161)

☐ Works as expected — notes:

In [ ]:
from hypertools import Pipeline

Xpipe = rng.normal(size=(60, 8))

# fit/transform/reuse with a two-step pipeline
pipe = Pipeline(['ZScore', 'PCA'])
out = pipe.fit_transform(Xpipe)
print("Pipeline(['ZScore','PCA']).fit_transform shape:", np.asarray(out).shape)
out_new = pipe.transform(rng.normal(size=(12, 8)))
print("Pipeline.transform (new data) shape:", np.asarray(out_new).shape)

# inverse_transform requires every step to be invertible. ZScore has no inverse
# (flagged in the QC task list, P1), so demonstrate the round-trip with a PCA-only pipeline.
inv_pipe = Pipeline(['PCA'])
z = inv_pipe.fit_transform(Xpipe)
back = inv_pipe.inverse_transform(z)
print("Pipeline(['PCA']).inverse_transform shape (expect back to (60, 8)):", np.asarray(back).shape)

**Verify:** fit_transform/transform/inverse_transform all run without error; inverse_transform returns data shaped (60, 8) again. — ☐ pass ☐ fail — notes:

## 9. Cross-module kwargs (GH #138)

☐ Works as expected — notes:

In [ ]:
# hyp.cluster(x, reduce=..., manip=...): manip -> reduce -> cluster all run in one call
Xcm = rng.normal(size=(60, 10))
labels_cm = hyp_cluster(Xcm, cluster='KMeans', n_clusters=3, reduce='PCA', ndims=3, manip='ZScore')
print("cluster(reduce=, manip=) labels (first 10):", np.asarray(labels_cm)[:10])

# hyp.reduce(x, align=...): align -> reduce all run in one call
datasets_cm = [rng.normal(size=(30, 6)) + rng.normal(scale=0.1, size=(30, 6)) for _ in range(2)]
reduced_cm = hyp_reduce(datasets_cm, reduce='PCA', ndims=2, align='HyperAlign')
print("reduce(align=) shapes:", [np.asarray(r).shape for r in reduced_cm])

**Verify:** cluster() with reduce=/manip= returns valid cluster labels; reduce() with align= aligns then reduces both datasets to (30, 2). — ☐ pass ☐ fail — notes:

## 10. Plot kwargs (GH #103/#154)

☐ Works as expected — notes:

In [ ]:
# label_alpha, xlabel/ylabel/zlabel
Xlk = rng.normal(size=(20, 3)).cumsum(axis=0)
labels_lk = [f"pt{i}" if i % 5 == 0 else None for i in range(20)]
hyp.plot(Xlk, ndims=3, labels=labels_lk, label_alpha=0.5,
         xlabel='dim 1', ylabel='dim 2', zlabel='dim 3',
         title='label_alpha + axis labels')

# animate= dict form (GH #154): mega-dict bundling style + related animation kwargs
hyp.plot(Xlk, ndims=3, animate={'style': 'spin', 'rotations': 1, 'duration': 5},
          title='animate= dict form (spin)', show=True)

**Verify:** The first plot shows every 5th point semi-transparently labeled with axes titled 'dim 1'/'dim 2'/'dim 3'; the second plot animates a spin using the animate= dict form (equivalent to animate='spin', rotations=1, duration=5). — ☐ pass ☐ fail — notes:

## 11. Animations (GH #123/#275)

☐ Works as expected — notes:

In [ ]:
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 50  # MB, keep inline HTML5 video small
from IPython.display import HTML, display

Xanim = rng.normal(size=(30, 3)).cumsum(axis=0)

anim_spin = hyp.plot(Xanim, ndims=3, animate='spin', duration=4, rotations=1, show=False)
display(HTML(anim_spin.to_html5_video()))

anim_window = hyp.plot(Xanim, ndims=3, animate='window', duration=4, focused=1, show=False)
display(HTML(anim_window.to_html5_video()))

**Verify:** Two inline videos play: a full 3D trajectory spinning (animate='spin'), and a moving windowed segment traveling along the trajectory (animate='window', focused=1). — ☐ pass ☐ fail — notes:

In [ ]:
# animate='morph': Hungarian point-cloud morph between 2+ datasets
Xmorph_a = rng.normal(size=(25, 3))
Xmorph_b = rng.normal(loc=[3, 3, 3], size=(25, 3))
anim_morph = hyp.plot([Xmorph_a, Xmorph_b], ndims=3, animate='morph', duration=4,
                       morph_samples=25, show=False)
display(HTML(anim_morph.to_html5_video()))

# 2D animation
Xanim2d = rng.normal(size=(30, 2)).cumsum(axis=0)
anim_2d = hyp.plot(Xanim2d, ndims=2, animate='parallel', duration=4, show=False)
display(HTML(anim_2d.to_html5_video()))

**Verify:** Points visibly morph/interpolate from cloud A's positions to cloud B's positions; the 2D animation plays correctly with no 3D-axes errors. — ☐ pass ☐ fail — notes:

## 12. Data loaders (hyp.load, GH #273/#116)

☐ Works as expected — notes:

In [ ]:
iris = hyp.load('iris')
print("iris:", type(iris).__name__, iris.shape, iris.columns.tolist())

penguins = hyp.load('penguins')
print("penguins:", type(penguins).__name__, penguins.shape, penguins.columns.tolist())

bechdel = hyp.load('fivethirtyeight/bechdel')
print("bechdel:", type(bechdel).__name__, bechdel.shape)
bechdel.head()

**Verify:** iris loads with sklearn-style columns ('sepal length (cm)', ... 'target'); penguins loads via seaborn with columns like 'species'/'bill_length_mm'; bechdel downloads with ~1794 rows. — ☐ pass ☐ fail — notes:

**Note (Kaggle loader):** `hyp.load('kaggle/uciml/iris')` requires the
optional `kagglehub` dependency (`pip install "hypertools[kaggle]"`, not
installed above) **and** Kaggle API credentials configured in the Colab
environment (`~/.kaggle/kaggle.json` or `KAGGLE_USERNAME`/`KAGGLE_KEY` env
vars) — not runnable in a bare Colab session without setup. Shape, if you
have credentials:

```python
!pip install -q "hypertools[kaggle]"
kaggle_iris = hyp.load('kaggle/uciml/iris')
print(kaggle_iris.shape)
```


## 13. Text (hyp.plot on strings, GH #198)

☐ Works as expected — notes:

In [ ]:
docs = [
    "The cat sat on the mat.",
    "Dogs are loyal companions.",
    "The stock market rallied today.",
    "Interest rates rose sharply this quarter.",
    "Kittens love to play with yarn.",
    "The Federal Reserve raised rates again.",
]
# Word2Vec doc vectors contain negative values, so pass semantic=None -- the default
# semantic model (LatentDirichletAllocation) rejects negative input (QC task list, P1).
hyp.plot(docs, ndims=2, vectorizer='Word2Vec', semantic=None,
         title='text plot, gensim Word2Vec vectorizer')

**Verify:** Six short documents plot as points in 2D using gensim's Word2Vec embedding (no ImportError, gensim extra was installed). — ☐ pass ☐ fail — notes:

## 14. Colorbar, surface, and density

☐ Works as expected — notes:

In [ ]:
Xcb = rng.normal(size=(50, 3)).cumsum(axis=0)
hyp.plot(Xcb, ndims=3, hue=np.linspace(0, 1, 50), colorbar=True, title='continuous hue + colorbar')

Xsurf = rng.normal(size=(200, 3))
hyp.plot(Xsurf, ndims=3, surface=True, title='surface=True')

Xdens = np.vstack([rng.normal(loc=[0, 0, 0], scale=0.5, size=(150, 3)),
                    rng.normal(loc=[4, 4, 4], scale=0.5, size=(150, 3))])
hyp.plot(Xdens, ndims=3, density=True, title='density=True')

**Verify:** Three plots render in order: a colorbar mapping continuous hue to color; a fitted surface/mesh through the point cloud; density shading highlighting the two dense blobs. — ☐ pass ☐ fail — notes:

## 15. LSL streaming (hyp.io.lsl_stream, GH #130)

☐ Works as expected — notes:

`hyp.io.lsl_stream()` resolves a **live** Lab Streaming Layer outlet on the
network and returns an infinite generator of samples — it has no synthetic/
offline mode, so it **cannot run in a plain Colab session** without a real
LSL outlet reachable on the same network (Colab's sandboxed VM generally
can't reach one anyway). Skip execution here; the code shape for local
verification (with a running LSL outlet, e.g. from `pylsl`'s
`example_outlet.py`) is:

```python
!pip install -q "hypertools[lsl]"
import hypertools as hyp

stream = hyp.io.lsl_stream(type='EEG', timeout=5.0)
hyp.plot(stream, stream_init=200, stream_chunk=20)
```


**Verify:** (Manual, off-Colab) Confirm lsl_stream() resolves a real outlet and hyp.plot streams/updates live. — ☐ pass ☐ fail — notes:

## Done

Once every section above has its checkbox ticked and notes filled in, save
this notebook (with outputs) as the QC record for the 1.0 release.